# 00 · Baseline: word + char TF-IDF → logistic regression

**Owner:** Allen
**Report section:** Model comparison (baseline row); Introduction

**Goal.** End-to-end reference pipeline: validation AUROC, 5-fold CV AUROC, saved predictions for ensembling, and a first Kaggle submission file.

Run with *Restart & Run All*. Set `QUICK = True` for a fast debug pass on the 2,000-row
starter file, `QUICK = False` for the numbers that go in the report. Every result uses the
shared split from `src.data.get_split()` (seed 42), so numbers are comparable across notebooks.

**Outputs:** `results/baseline.csv`, `results/preds_baseline.csv`, `figures/roc_baseline.png`, `submissions/submission_*_baseline.csv`.

In [ ]:
# --- setup: make `src/` importable from anywhere (repo root, notebooks/, Kaggle/Colab clone) ---
import sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

QUICK = True  # True: 2,000-row starter file for fast debugging. False: full data (report numbers).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import SEED, set_seed, RESULTS_DIR, FIGURES_DIR
from src.data import get_split, load_test, load_unlabeled, load_train, get_cv_folds, xy
from src.features import build_features
from src.models import make_model
from src.evaluate import (auroc, cross_val_auroc, plot_roc, savefig, log_result,
                          save_predictions, predict_proba1)

set_seed(SEED)
SUFFIX = "_quick" if QUICK else ""   # keeps QUICK-mode outputs separate from real ones
from src.evaluate import write_submission

In [ ]:
train_df, val_df = get_split(quick=QUICK)
test_df = load_test()
X_tr, y_tr = xy(train_df)
X_va, y_va = xy(val_df)

## 1. Fit on the train split, score the validation split

In [ ]:
model = make_model("baseline")
print(model)

t0 = time.time()
model.fit(X_tr, y_tr)
fit_s = time.time() - t0
val_probs = predict_proba1(model, X_va)
val_auc = auroc(y_va, val_probs)
n_feats = model.named_steps["features"].transform(X_va[:1]).shape[1]
print(f"val AUROC = {val_auc:.4f}   fit time = {fit_s:.1f}s   features = {n_feats:,}")
plot_roc(y_va, val_probs, "baseline" + SUFFIX, title="Baseline ROC (validation split)");

## 2. 5-fold cross-validated AUROC on all labelled data

Stratified folds (seed 42); exact-duplicate texts are kept in the same fold.

In [ ]:
full_df = pd.concat([train_df, val_df], ignore_index=True)
t0 = time.time()
cv = cross_val_auroc(make_model("baseline"), full_df["text"], full_df["label"], verbose=True)
cv_s = time.time() - t0
print(f"5-fold CV AUROC = {cv['cv_auroc_mean']:.4f} ± {cv['cv_auroc_std']:.4f}   ({cv_s:.0f}s)")

## 3. Refit on all labelled data → test predictions → submission

In [ ]:
final = make_model("baseline").fit(full_df["text"], full_df["label"])
test_probs = predict_proba1(final, test_df["text"])

save_predictions("baseline" + SUFFIX, val_df, val_probs, test_df, test_probs)
sub_path = write_submission(test_df, test_probs, tag="baseline" + SUFFIX)
pd.read_csv(sub_path).describe()

In [ ]:
log_result("baseline", config={"model": "baseline", "features": "word+char", "clf": "logreg C=1",
                              "quick": QUICK, "n_train": len(train_df), "n_val": len(val_df)},
           metrics={"val_auroc": val_auc, **cv, "fit_seconds": round(fit_s, 1),
                    "cv_seconds": round(cv_s, 1), "n_features": n_feats})

## Notes

* Test probabilities come from the model refit on *all* labelled rows. Validation
  probabilities come from the train-split model, so they are honest held-out scores.
* `test.csv` contains styles and domains not seen in train, so validation AUROC is likely
  optimistic. See `21_generalization.ipynb`.